# Agentic AI: the loop, the failure modes, and the eval

**GDG Mississauga — 30 July 2026 · 45 min · hands on**

## What this project is
One idea, five parts: **an agent is a `while` loop that decides when to stop.**
Everything else is plumbing.

You work against a toy hardware-store catalog (20 rows, `data/store.csv`) and:

1. **Build** the loop by hand — no framework, ~40 lines.
2. **Rebuild** it on Google ADK, and see what the framework took over.
3. **Break** it three ways on purpose.
4. **Score** it — final-answer accuracy vs. trajectory accuracy.
5. **Decide** when an agent is the right tool at work.

## Before you start
| | |
|---|---|
| **No API key needed** | `MODE = "mock"` runs a deterministic fake model locally. No network, no credentials, no bill. |
| **Run Cell 0 first** | Wait for the `YOU'RE READY` banner. |
| **Cells are independent** | Typo? Fix it and re-run just that cell. |

## Roadmap
| Part | Topic | You do | Time |
|---|---|---|---|
| 1 | The loop, no framework | read + run | 5–12 min |
| 2 | The same agent in ADK | **Exercise 1** | 12–24 min |
| 3 | Breaking it on purpose | **Exercises 2 & 3** | 24–32 min |
| 4 | Eval: answer vs. trajectory | run | 32–40 min |
| 5 | Where agents actually work | discussion | 40–45 min |

Short on time? The presenter calls out ⏭️ **`CUT AT 30`** (skip it) and
👀 **`WATCH ONLY`** (presenter runs it, you watch).

## Cell 0 — Setup

**Run this once, before anything else.** Run it again if something breaks later.

**What it does:** detects Colab vs. local → restores the support files → loads the
catalog → resolves the backend (mock or live) → defines the helpers and tools
every later cell uses → makes one test call.

**Success = a `YOU'RE READY` banner.** Anything else names the specific fix; this
cell prints no raw tracebacks, by design.

In [1]:
# ===========================================================================
# CELL 0 — SETUP.  Run this first; run it again whenever something breaks.
#
# What this cell does, in the order the sections appear below:
#   1. finds the support files (mock_llm.py, eval_set.json, data/store.csv)
#   2. installs the pinned packages, on Colab only
#   3. loads the 20-row catalog into STORE
#   4. resolves the backend -> MODE and MODEL
#   5. defines raw_model_call()               <- Part 1's hand-written loop
#   6. defines make_agent() / run_and_trace() <- Parts 2, 3 and 4
#   7. defines the six store tools            <- every part
#   8. makes one test call and prints YOU'RE READY
#
# No other cell in this notebook depends on anything except this one.
# ===========================================================================

# Inline copy of data/store.csv, used only if the real file is missing (step 1).
_STORE_CSV_FALLBACK = '''sku,product_name,category,price,stock_qty,aisle
HD-1001,16 oz Claw Hammer,hand tools,14.99,42,A1
HD-1002,25 ft Tape Measure,hand tools,11.49,63,A1
HD-1003,Adjustable Wrench 10 in,hand tools,18.25,27,A2
HD-1004,Phillips Screwdriver Set 6 pc,hand tools,22.00,15,A2
HD-1005,Utility Knife Retractable,hand tools,7.99,88,A3
HD-2001,Cordless Drill 20V,power tools,129.00,12,B1
HD-2002,Circular Saw 7-1/4 in,power tools,149.50,6,B1
HD-2003,Orbital Sander 5 in,power tools,68.75,9,B2
HD-2004,Shop Vacuum 6 gal,power tools,89.99,4,B2
HD-3001,Exterior Latex Paint 1 gal White,paint,38.49,54,C1
HD-3002,Interior Primer 1 gal,paint,26.99,31,C1
HD-3003,Paint Roller Kit 9 in,paint,13.75,72,C2
HD-3004,Painters Tape 1.88 in,paint,6.49,120,C2
HD-4001,Wood Screws 2 in 100 ct,fasteners,8.95,145,D1
HD-4002,Drywall Anchors 50 ct,fasteners,5.49,98,D1
HD-4003,Carriage Bolts 3 in 25 ct,fasteners,12.30,37,D2
HD-5001,LED Shop Light 4 ft,electrical,34.99,23,E1
HD-5002,Extension Cord 50 ft,electrical,42.00,18,E1
HD-5003,GFCI Outlet 15A,electrical,19.99,64,E2
HD-6001,PVC Pipe 2 in x 10 ft,plumbing,16.80,45,F1
'''

MODE = "mock"   # "mock" (default, no credentials, no network) | "live" (presenter demo)

# ---------------------------------------------------------------------------
# Model IDs live HERE and nowhere else in this notebook.
#
# Verified 2026-07-26 against ai.google.dev/gemini-api/docs/models (and
# /deprecations) and docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/models.
#
# Why gemini-3.5-flash: it is GA, supports function calling, is cheap and fast,
# and has NO announced shutdown date (released 2026-05-19) — so this notebook
# still works months from now. We avoid the "gemini-flash-latest" alias because
# it hot-swaps versions, and a rehearsed demo should be boring and reproducible.
# We avoid gemini-2.5-flash: earliest shutdown 2026-10-16.
# Cheaper/faster swap: "gemini-3.5-flash-lite".
MODEL_VERTEX = "gemini-3.5-flash"      # Vertex AI (ADC) backend
MODEL_AI_STUDIO = "gemini-3.5-flash"   # AI Studio (API key) backend
# NOTE: these two happen to be the same string today. They get separate
# constants because the valid ID sets genuinely differ between backends —
# AI Studio accepts "-latest" aliases that Vertex does not reliably serve.

# ---------------------------------------------------------------------------
# Support files this notebook needs next to it. Set this if you forked the repo.
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/gdg-agentic-ai-workshop"
PINNED = ["google-adk==2.5.0", "google-genai==2.14.0"]

import os, sys, csv, json, logging, warnings, pathlib

MODE = os.environ.get("WORKSHOP_MODE", MODE)  # automation hook; ignore this line

IN_COLAB = "google.colab" in sys.modules
HERE = pathlib.Path.cwd()
NEEDED = ["mock_llm.py", "eval_set.json", "data/store.csv"]

# ADK logs an EXPERIMENTAL notice about JSON schema generation for tools. It is
# internal noise nobody in this room can act on, so we hide it to keep output
# readable. Nothing else is suppressed.
warnings.filterwarnings("ignore", message=".*JSON_SCHEMA_FOR_FUNC_DECL.*")

# ADK logs a full async traceback for conditions we deliberately trigger and
# handle ourselves — notably hitting the max_llm_calls cap in Part 3, which is
# the cap WORKING. A 40-frame traceback on a projector teaches nothing, and we
# print a plain-language cause for every failure via explain_error() below.
# Note the underscore: ADK's loggers are named "google_adk.<module>", not
# "google.adk.<module>". Verified against google-adk 2.5.0.
logging.getLogger("google_adk").setLevel(logging.CRITICAL)
logging.getLogger("google_genai").setLevel(logging.ERROR)

# --- 1. Get the support files -------------------------------------------------
def _missing():
    return [f for f in NEEDED if not (HERE / f).exists()]

if _missing() and IN_COLAB:
    # Colab opens a bare .ipynb, so the repo's small support files are absent.
    print(f"Colab detected. Fetching support files for: {', '.join(_missing())}")
    os.system(f"git clone --depth 1 -q {REPO_URL} /content/_ws 2>/dev/null")
    for f in NEEDED:
        src, dst = pathlib.Path("/content/_ws") / f, HERE / f
        if src.exists() and not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(src.read_bytes())

if (HERE / "data/store.csv").exists() is False:
    # Last-resort fallback: the dataset is 20 rows, so we can just inline it
    # rather than let a network failure end the workshop.
    (HERE / "data").mkdir(exist_ok=True)
    (HERE / "data/store.csv").write_text(_STORE_CSV_FALLBACK)
    print("Used the built-in copy of data/store.csv (no download needed).")

# --- 2. Install ADK if needed (Colab only; local users ran `uv sync`) --------
def _have_adk():
    try:
        import google.adk  # noqa: F401
        return True
    except ImportError:
        return False

if not _have_adk() and IN_COLAB:
    print("Installing pinned packages (about 30 seconds)...")
    os.system(f"{sys.executable} -m pip install -q " + " ".join(PINNED))

ADK_AVAILABLE = _have_adk()

# --- 3. Load the dataset (stdlib csv; no pandas needed) ----------------------
def load_store(path="data/store.csv"):
    """Read the toy catalog. Typed on the way in so tools can do arithmetic."""
    with open(path, newline="") as fh:
        rows = list(csv.DictReader(fh))
    for r in rows:
        r["price"] = float(r["price"])
        r["stock_qty"] = int(r["stock_qty"])
    return rows

STORE = load_store()

# --- 4. Resolve the backend --------------------------------------------------
# This is the ONLY place in the notebook that branches on MODE. Every cell below
# is byte-identical in both modes: they all just use MODEL and raw_model_call.
def _redact(project: str) -> str:
    """Project IDs go on a projector in a room full of strangers."""
    if not project:
        return "unset"
    return project[:3] + "..." + project[-2:] if len(project) > 6 else "***"

def _resolve_live():
    """Try Vertex/ADC, then AI Studio. Returns (model_id, description) or None.

    Silent and non-interactive by design: never prompts, never prints a secret.
    """
    # Backend 1: Vertex AI via Application Default Credentials.
    try:
        import google.auth
        creds, adc_project = google.auth.default()
        project = os.environ.get("GOOGLE_CLOUD_PROJECT") or adc_project
        if project:
            location = os.environ.get("GOOGLE_CLOUD_LOCATION", "global")
            # google-genai reads these three env vars. Accepted truthy values
            # are "true"/"TRUE"/"True"/"1" (it lowercases and tests membership).
            os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
            os.environ["GOOGLE_CLOUD_PROJECT"] = project
            os.environ["GOOGLE_CLOUD_LOCATION"] = location
            return MODEL_VERTEX, (
                f"Vertex AI, project={_redact(project)}, "
                f"location={location}, model={MODEL_VERTEX}"
            )
    except Exception:
        pass  # no ADC on this machine; fall through to the key path

    # Backend 2: AI Studio API key, only if one is already in the environment.
    if os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"):
        os.environ.pop("GOOGLE_GENAI_USE_VERTEXAI", None)
        return MODEL_AI_STUDIO, f"AI Studio API key, model={MODEL_AI_STUDIO}"

    return None

MODEL = None
if MODE == "live":
    if not ADK_AVAILABLE:
        print("MODE=live needs google-genai installed. Falling back to mock.")
        MODE = "mock"
    else:
        resolved = _resolve_live()
        if resolved is None:
            # One actionable line, then degrade instead of crashing.
            print("MODE=live but no credentials resolved. Falling back to mock.")
            print("  Fix: run `gcloud auth application-default login` and")
            print("       `gcloud config set project YOUR_PROJECT_ID`,")
            print("       or set GOOGLE_API_KEY for the AI Studio path.")
            MODE = "mock"
        else:
            MODEL, _desc = resolved
            print(f"MODE=live — {_desc}")

if MODE == "mock":
    import mock_llm
    MODEL = mock_llm.MockLlm() if ADK_AVAILABLE else "mock-deterministic-1"
    print("MODE=mock — deterministic fake model, no API calls.")

# --- 5. The raw (no-framework) model call used by Part 1 ---------------------
def raw_model_call(question, schemas, history):
    """One model turn, no framework. Returns a plain dict describing the choice.

    Mock and live both return the same shape:
      {"kind": "tool_call", "name": str, "args": dict, "why": str}
      {"kind": "final",     "text": str,                "why": str}
    """
    if MODE == "mock":
        import mock_llm
        d = mock_llm.decide(question, schemas, history)
        top = d["ranked"][0] if d["ranked"] else None
        why = (
            f"best match {top['name']} scored {top['score']:.1f} on words {top['matched']}"
            if top else "no tools available"
        )
        if d["kind"] == "tool_call":
            return {"kind": "tool_call", "name": d["name"], "args": d["args"], "why": why}
        # `why` explains the CHOICE. It cannot explain the STOP, and reporting it
        # as if it could is how a trace ends up lying to the room: the ranking is
        # a pure function of the question and the tool list, so it is IDENTICAL on
        # every step of a run, and a tool can score top marks and still be the
        # wrong next step. decide() reports the branch it actually left on.
        return {"kind": "final", "text": d["text"], "why": why,
                "stop_reason": d["stop_reason"]}

    # Live: talk to Gemini directly through google-genai. No ADK here on purpose.
    from google import genai
    from google.genai import types
    client = genai.Client()
    contents = []
    for turn in history:
        if turn["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=turn["content"])]))
        elif turn["role"] == "assistant":
            contents.append(types.Content(role="model", parts=[types.Part(
                function_call=types.FunctionCall(name=turn["name"], args=turn["args"]))]))
        elif turn["role"] == "tool":
            contents.append(types.Content(role="user", parts=[types.Part(
                function_response=types.FunctionResponse(
                    name=turn["name"], response={"result": turn["content"]}))]))
    decls = [types.FunctionDeclaration(
        name=s["name"], description=s["description"],
        parameters_json_schema={"type": "object", "properties": s["parameters"]["properties"]},
    ) for s in schemas]
    resp = client.models.generate_content(
        model=MODEL, contents=contents,
        config=types.GenerateContentConfig(
            tools=[types.Tool(function_declarations=decls)],
            temperature=0,  # determinism matters more than flair in a demo
        ),
    )
    for part in (resp.candidates[0].content.parts or []):
        if part.function_call:
            return {"kind": "tool_call", "name": part.function_call.name,
                    "args": dict(part.function_call.args or {}), "why": "model chose a tool"}
    return {"kind": "final", "text": resp.text or "(no answer)", "why": "model answered directly",
            "stop_reason": "the model answered instead of calling another tool"}

# --- 6. Shared ADK helpers (re-shown with commentary in Part 2) --------------
if ADK_AVAILABLE:
    from google.adk.agents import LlmAgent
    from google.adk.agents.invocation_context import LlmCallsLimitExceededError
    from google.adk.agents.run_config import RunConfig
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai import types as genai_types

    def explain_error(exc) -> str:
        """Turn an exception into one line a human can act on.

        Nobody learns anything from a 40-frame async traceback on a projector,
        and an attendee on bad wifi needs to know which of these it is.
        """
        text = f"{type(exc).__name__}: {exc}"
        low = text.lower()
        if isinstance(exc, LlmCallsLimitExceededError):
            return "Hit the max_llm_calls cap. That is the cap doing its job."
        if "api key" in low or "api_key" in low or "unauthenticated" in low:
            return "Bad or missing API key. Check GOOGLE_API_KEY, or use MODE='mock'."
        if "default credentials" in low or "could not automatically determine" in low:
            return ("No Application Default Credentials. Run "
                    "`gcloud auth application-default login`, or use MODE='mock'.")
        if "permission" in low or "403" in low:
            return ("Permission denied on the project. You need roles/aiplatform.user, "
                    "and the Vertex AI API enabled.")
        if "quota" in low or "429" in low or "resource_exhausted" in low:
            return "Rate limited or out of quota. Wait a minute, or use MODE='mock'."
        if "not found" in low or "404" in low:
            return f"Model not found for this backend/region. Check MODEL (currently {MODEL!r})."
        if any(w in low for w in ("connection", "timeout", "dns", "network", "ssl")):
            return "Network failure. Conference wifi. Set MODE='mock' and carry on."
        return text

    def make_agent(tools, name="store_assistant", instruction=None):
        """Build a fresh ADK agent. `model=MODEL` is the whole mock/live seam:
        ADK accepts `model: Union[str, BaseLlm]`, so a fake model drops straight in."""
        return LlmAgent(
            name=name,
            model=MODEL,
            instruction=instruction or (
                "You answer questions about a hardware store's inventory. "
                "Use the provided tools. Do not guess at data."
            ),
            tools=tools,
        )

    async def run_and_trace(agent, question, max_llm_calls=5, show=True):
        """Run an ADK agent and print its event stream in Part 1's format.

        Returns (final_text, trajectory, total_tokens) so later cells can score.
        """
        session_service = InMemorySessionService()
        await session_service.create_session(app_name="ws", user_id="u", session_id="s")
        runner = Runner(app_name="ws", agent=agent, session_service=session_service)

        final_text, trajectory, tokens, step = "", [], 0, 0
        if show:
            print(f"Q: {question}")

        # Every model call in this notebook is wrapped. On failure we print one
        # actionable line and return what we have, rather than dumping a
        # traceback and losing the partial trace that explains what happened.
        try:
            async for event in runner.run_async(
                user_id="u", session_id="s",
                new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=question)]),
                run_config=RunConfig(max_llm_calls=max_llm_calls),
            ):
                if event.usage_metadata and event.usage_metadata.total_token_count:
                    tokens += event.usage_metadata.total_token_count
                for call in event.get_function_calls():
                    step += 1
                    trajectory.append(call.name)
                    if show:
                        print(f"  STEP {step}")
                        print(f"    TOOL_CALL   {call.name}({dict(call.args or {})})")
                for resp in event.get_function_responses():
                    if show:
                        print(f"    OBSERVATION {resp.response}")
                if event.is_final_response() and event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            final_text = part.text
        except Exception as exc:
            final_text = f"[stopped] {explain_error(exc)}"
            if show:
                print(f"  STOPPED     {explain_error(exc)}")

        if show:
            print(f"  ANSWER      {final_text}")
            print(f"  [{len(trajectory)} tool call(s), ~{tokens} tokens]")
        return final_text, trajectory, tokens

# --- 7. The store tools -----------------------------------------------------
# Defined here so that ANY part of this notebook runs standalone after Cell 0 —
# nobody should have to hunt for which earlier cell they forgot to run. Part 2
# re-derives these same four with commentary, which is where the interesting
# discussion about docstrings lives. Redefining them there is harmless.

def get_product_details(sku: str) -> dict:
    """Return the full record for one product given its SKU: product name, category, price, stock quantity and aisle."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", **row}
    return {"status": "error", "message": f"no product with SKU {sku}"}

def check_stock_quantity(sku: str) -> dict:
    """Report how many units of one product remain in stock right now, given its SKU."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", "sku": row["sku"], "units_in_stock": row["stock_qty"]}
    return {"status": "error", "message": f"no product with SKU {sku}"}

def list_products_in_category(category: str) -> dict:
    """List every product in a given department category, such as paint or plumbing or fasteners."""
    names = [r["product_name"] for r in STORE if r["category"] == category.lower()]
    return {"status": "success", "products": ", ".join(names) or "none found"}

def calculate_order_total(sku: str, quantity: int) -> dict:
    """Multiply a product's unit price by a quantity to get the order total in dollars."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", "order_total_usd": round(row["price"] * quantity, 2)}
    return {"status": "error", "message": f"no product with SKU {sku}"}

def find_product_by_name(product_name: str) -> dict:
    """Search the store catalog for products whose name contains the given text. Returns matching SKUs and product names."""
    hits = [f"{r['sku']} {r['product_name']}" for r in STORE
            if product_name.lower() in r["product_name"].lower()]
    return {"status": "success", "matches": "; ".join(hits) or "none found"}

def total_inventory_value(category: str) -> dict:
    """Calculate the total dollar value of all inventory currently in stock for a category, by multiplying price by stock quantity for every product."""
    rows = [r for r in STORE if r["category"] == category.lower()]
    return {"status": "success",
            "total_value_usd": round(sum(r["price"] * r["stock_qty"] for r in rows), 2)}

# The four the agent starts with in Part 2...
STORE_TOOLS = [get_product_details, check_stock_quantity,
               list_products_in_category, calculate_order_total]
# ...and the wider set Part 4 evaluates against.
EVAL_TOOLS = STORE_TOOLS + [find_product_by_name, total_inventory_value]

# --- 8. Prove it works ------------------------------------------------------
try:
    _probe = raw_model_call(
        "What is the price of SKU HD-2001?",
        [{"name": "get_product_details",
          "description": "Return the price and stock quantity for a product given its SKU.",
          "parameters": {"properties": {"sku": {"type": "string"}}}}],
        [{"role": "user", "content": "What is the price of SKU HD-2001?"}],
    )
    assert _probe["kind"] in ("tool_call", "final")
    print()
    print("=" * 62)
    print(f"  YOU'RE READY     MODE={MODE}   rows loaded={len(STORE)}")
    print(f"  Parts 2-4 (ADK): {'available' if ADK_AVAILABLE else 'UNAVAILABLE - Part 1 still works'}")
    print("=" * 62)
except FileNotFoundError as e:
    print(f"MISSING FILE: {e.filename}")
    print("  Fix: make sure this notebook sits next to mock_llm.py and data/store.csv.")
except Exception as e:
    print(f"NOT READY: {type(e).__name__}: {e}")
    print("  Most likely causes, in order:")
    print("   1. MODE='live' without credentials -> set MODE='mock' and re-run.")
    print("   2. mock_llm.py not next to this notebook.")
    print("   3. Packages missing -> run `uv sync` locally, or re-run this cell in Colab.")

MODE=mock — deterministic fake model, no API calls.

  YOU'RE READY     MODE=mock   rows loaded=20
  Parts 2-4 (ADK): available


### What the mock model actually is — read this

`MODE = "mock"` does **not** call Gemini. No key, no network, no cloud account. It
is a deterministic stand-in in `mock_llm.py` that picks a tool like this:

> Tokenize the question → drop stopwords → count word overlap with each tool's
> **name and docstring**. A tool is eligible only if **≥2** question words appear
> in its docstring. Highest score wins; ties break alphabetically. Nothing clears
> the bar → it answers in text instead.

| Real, not simulated | Faked — know the difference |
|---|---|
| The loop: decide → act → observe → stop | **No language understanding.** It matches words, not meaning. |
| Tool wiring, schema generation from your type hints, ADK events | **No reasoning.** Multi-step behaviour comes from the loop, not from planning. |
| The Part 3 failure modes | A docstring written in **synonyms** misses here where Gemini would hit. |
| A real eval harness, with deterministic scores | Answers are templated (`"Based on <tool>: <result>"`), not generated prose. |

Every run prints `MODE=mock — deterministic fake model, no API calls.`

---
# Part 1 — The loop, with no framework at all
**5–12 min · read and run · do not edit**

## What this part does
The whole idea in standard-library Python: a `while` loop, a dict of tools,
function-calling round trips, and a `max_steps` stop condition.

## First — where an agent sits
| | Picks the next step | Handles an input you didn't anticipate? |
|---|---|---|
| **Single model call** | nobody — text in, text out | No. Only as current as its training data. |
| **RAG** | you — one retrieval, always the same shape | No. |
| **Fixed workflow** | you — A, then B, then C | No, but predictable and debuggable. |
| **Agent** | **the model**, including when to stop | Yes — at the cost of determinism, a bounded bill, and easy debugging. |

Reach for the workflow first. Reach for the agent when you genuinely cannot
enumerate the steps in advance.

## What the cell below does
Defines 2 tools → turns them into model-visible schemas with `describe()` → runs
`agent_loop()`. Watch the `STEP / THOUGHT / TOOL_CALL / OBSERVATION` trace.

In [2]:
# ===========================================================================
# PART 1 — the agent loop, hand-written. No framework anywhere in this cell.
#
#   describe()    turns a Python function into the schema the model sees
#   agent_loop()  decide -> act -> observe -> decide again -> stop
#
# In mock mode this needs nothing but the standard library and mock_llm.py, so
# it survives a failed `pip install`.
# ===========================================================================

import inspect

def get_product_details(sku: str) -> dict:
    """Return the full record for one product given its SKU: product name, category, price, stock quantity and aisle."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return dict(row)
    return {"error": f"no product with SKU {sku}"}

def check_stock_quantity(sku: str) -> dict:
    """Report how many units of one product remain in stock right now, given its SKU."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"sku": row["sku"], "units_in_stock": row["stock_qty"]}
    return {"error": f"no product with SKU {sku}"}

TOOLS = {fn.__name__: fn for fn in (get_product_details, check_stock_quantity)}

def describe(fn):
    """Turn a Python function into the tool schema a model sees.

    Read this and you have read the interesting half of every agent framework:
    it introspects your signature and hands your docstring to the model.
    """
    py_to_json = {str: "string", int: "integer", float: "number", bool: "boolean"}
    return {
        "name": fn.__name__,
        "description": inspect.getdoc(fn) or "",
        "parameters": {"properties": {
            p.name: {"type": py_to_json.get(p.annotation, "string")}
            for p in inspect.signature(fn).parameters.values()
        }},
    }

SCHEMAS = [describe(fn) for fn in TOOLS.values()]


def agent_loop(question, max_steps=4):
    """The loop. Decide -> act -> observe -> decide again -> stop."""
    history = [{"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        decision = raw_model_call(question, SCHEMAS, history)   # 1. DECIDE
        print(f"STEP {step}")
        print(f"  THOUGHT     {decision['why']}")

        if decision["kind"] == "final":                          # 2. STOP?
            # Two separate facts, so they get two separate lines. THOUGHT is the
            # same every step; STOP is the one that changed and ended the loop.
            print(f"  STOP        {decision['stop_reason']}")
            print(f"  ANSWER      {decision['text']}")
            return decision["text"]

        name, args = decision["name"], decision["args"]
        print(f"  TOOL_CALL   {name}({args})")
        result = TOOLS[name](**args)                             # 3. ACT
        print(f"  OBSERVATION {result}")                         # 4. OBSERVE

        history.append({"role": "assistant", "name": name, "args": args})
        history.append({"role": "tool", "name": name, "content": str(result),
                        "fields": result, "args": args})

    # The stop condition is not optional. Without it this is a while True.
    print(f"  STOP        hit max_steps={max_steps}")
    return "Stopped: step limit reached."


agent_loop("How many units of SKU HD-2002 are left in stock?")

STEP 1
  THOUGHT     best match check_stock_quantity scored 5.0 on words ['stock', 'many', 'sku', 'unit']
  TOOL_CALL   check_stock_quantity({'sku': 'HD-2002'})
  OBSERVATION {'sku': 'HD-2002', 'units_in_stock': 6}
STEP 2
  THOUGHT     best match check_stock_quantity scored 5.0 on words ['stock', 'many', 'sku', 'unit']
  STOP        nothing left to chain: get_product_details would re-read the question, not the result
  ANSWER      Based on check_stock_quantity: {'sku': 'HD-2002', 'units_in_stock': 6}


"Based on check_stock_quantity: {'sku': 'HD-2002', 'units_in_stock': 6}"

### What just happened

- The loop asked the model **what to do next** — it did not tell it.
- The model returned a **tool name and arguments**, not prose. *Your* code ran the
  function; the model never touches your data.
- The result went into `history`, and the model was asked again.
- It stopped because it decided it had enough — but `max_steps` was there in case
  it didn't. **Delete that line and you have `while True` on a billing account.**

Parts 2–4 add a framework, break it, and measure it. Nothing ahead is
conceptually bigger than this.

---
# Part 2 — The same agent, in Google ADK
**12–24 min**

## What this part does
Rebuilds the *identical* agent on a framework — same tools, same question, same
trace format — so the only thing you notice is what ADK took over. Then you add a
tool of your own in **Exercise 1**.

## The one thing to take away
Tools are **plain Python functions with type hints and a docstring.** No
decorator, no registration, no wrapper class. ADK reads your signature for the
parameter schema and your **docstring for the tool description.**

> ### The docstring IS the prompt.
> The sentence you wrote for the next human is what the model reads when it
> decides whether to call your function. Not documentation sitting near your code
> — production prompt text, in your repo, going through code review.

## What the cell below does
Re-defines the four store tools → wraps them in an ADK agent via `make_agent()` →
asks Part 1's question. No hand-written loop this time.

In [3]:
# ===========================================================================
# PART 2 — the same agent, rebuilt on ADK. Compare to Part 1 line for line:
# the loop, the history list and the schema builder are all gone.
#
# Tools are PLAIN PYTHON FUNCTIONS. ADK reads your type hints to build the
# parameter schema and your docstring to build the tool description. Verified
# against google-adk 2.5.0: passing a bare function into `tools=[...]` makes ADK
# wrap it in a FunctionTool for you. No decorator, no registry, no wrapper.
# ===========================================================================

def get_product_details(sku: str) -> dict:
    """Return the full record for one product given its SKU: product name, category, price, stock quantity and aisle."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", **row}
    return {"status": "error", "message": f"no product with SKU {sku}"}

def check_stock_quantity(sku: str) -> dict:
    """Report how many units of one product remain in stock right now, given its SKU."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", "sku": row["sku"], "units_in_stock": row["stock_qty"]}
    return {"status": "error", "message": f"no product with SKU {sku}"}

def list_products_in_category(category: str) -> dict:
    """List every product in a given department category, such as paint or plumbing or fasteners."""
    names = [r["product_name"] for r in STORE if r["category"] == category.lower()]
    return {"status": "success", "products": ", ".join(names) or "none found"}

def calculate_order_total(sku: str, quantity: int) -> dict:
    """Multiply a product's unit price by a quantity to get the order total in dollars."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", "order_total_usd": round(row["price"] * quantity, 2)}
    return {"status": "error", "message": f"no product with SKU {sku}"}


STORE_TOOLS = [get_product_details, check_stock_quantity,
               list_products_in_category, calculate_order_total]

agent = make_agent(STORE_TOOLS)

# `run_and_trace` was defined in Cell 0 so every part of this notebook can be
# run on its own. It prints ADK's event stream in Part 1's exact format:
# the events ARE the loop you just wrote by hand.
await run_and_trace(agent, "How many units of SKU HD-2002 are left in stock?")

Q: How many units of SKU HD-2002 are left in stock?
  STEP 1
    TOOL_CALL   check_stock_quantity({'sku': 'HD-2002'})
    OBSERVATION {'status': 'success', 'sku': 'HD-2002', 'units_in_stock': 6}
  ANSWER      Based on check_stock_quantity: sku=HD-2002, units_in_stock=6
  [1 tool call(s), ~48 tokens]


('Based on check_stock_quantity: sku=HD-2002, units_in_stock=6',
 ['check_stock_quantity'],
 48)

### What the framework just did for you

You deleted the loop. Here is the exact trade:

| You hand-wrote in Part 1 | ADK handles |
|---|---|
| `history`, appended by hand | **Sessions** — via `InMemorySessionService`; swap for a database in production, same interface |
| passing `history` into every call | **State** — carried across turns and tool calls |
| `describe()` — introspecting signatures | **Tool schema generation** from your type hints and docstring |
| `print()` at each step | **Streaming events** — a typed async stream to inspect, log, or forward to a UI |
| nothing — you had no hooks | **Callbacks** — `before_model_callback`, `after_tool_callback`, … for guardrails and logging |
| `for step in range(max_steps)` | **`RunConfig(max_llm_calls=...)`** — the same stop condition, enforced by the runtime |

**What it did not do: decide anything for you.** Tool choice is still driven
entirely by text *you* wrote — which is why Part 3 breaks it so easily.

**Honest note:** you traded 40 readable lines for a dependency, an async event
model, and stack traces through someone else's code. For one agent, Part 1 may be
the better engineering call. The framework pays for itself at sessions,
persistence, callbacks, and multi-agent routing.

### 🔧 EXERCISE 1 — add your own tool
**~6 min · your first hands-on task**

**Goal:** one function, so the agent can answer a question it currently refuses —
*"What is the total value of the paint inventory we have in stock?"*

**The cell below:** runs the failing question → shows a worked example
(`count_products_in_category`) → hands you a stub with three `YOUR CODE HERE`
blanks → re-runs the question with your tool attached.

**The blanks:** 1. a docstring sentence · 2. `row["price"] * row["stock_qty"]` ·
3. already filled in for you.

**What to actually watch:** write the docstring *first*, as a real sentence using
the words someone would type in the question. Then make it vague — `"Gets the
value."` — and re-run. Your tool stops being called. Same code, same name, same
signature.

In [ ]:
# ===========================================================================
# EXERCISE 1 — SOLUTION
#
# Three runs, so you can see the docstring doing the work:
#   1. before: no tool for the question, so the agent declines
#   2. after:  the same question with total_inventory_value attached
#   3. again:  identical code and name, VAGUE docstring -> not called
# ===========================================================================
# RUN 1 — before.
await run_and_trace(make_agent(STORE_TOOLS),
                    "What is the total value of the paint inventory we have in stock?")

print("\n" + "-" * 62 + "\n")

def total_inventory_value(category: str) -> dict:
    """Calculate the total dollar value of all inventory currently in stock for a category, by multiplying price by stock quantity for every product."""
    rows = [r for r in STORE if r["category"] == category.lower()]
    total = 0.0

    return {"status": "success", "total_value_usd": round(total, 2)}


MY_TOOLS = STORE_TOOLS + [total_inventory_value]

# RUN 2 — after.
await run_and_trace(make_agent(MY_TOOLS),
                    "What is the total value of the paint inventory we have in stock?")

# Check by hand, because you always should:
#   38.49*54 + 26.99*31 + 13.75*72 + 6.49*120
#   = 2078.46 + 836.69 + 990.00 + 778.80 = 4683.95
print("\nhand-checked expected value: 4683.95")

# ---------------------------------------------------------------------------
# RUN 3 — the lesson, made explicit. Same function, same name, same code.
# Only the docstring changes. Watch it stop being called.
# ---------------------------------------------------------------------------
def total_inventory_value_vague(category: str) -> dict:
    """Gets the value."""
    rows = [r for r in STORE if r["category"] == category.lower()]
    return {"status": "success",
            "total_value_usd": round(sum(r["price"] * r["stock_qty"] for r in rows), 2)}

total_inventory_value_vague.__name__ = "total_inventory_value"   # identical name

print("\n" + "-" * 62)
print("Same tool, vague docstring:\n")
await run_and_trace(make_agent(STORE_TOOLS + [total_inventory_value_vague]),
                    "What is the total value of the paint inventory we have in stock?")

Q: What is the total value of the paint inventory we have in stock?
  ANSWER      I don't have a tool that can answer that.
  [0 tool call(s), ~26 tokens]

--------------------------------------------------------------

Q: What is the total value of the paint inventory we have in stock?
  STEP 1
    TOOL_CALL   total_inventory_value({'category': 'paint'})
    OBSERVATION {'status': 'success', 'total_value_usd': 0.0}
  ANSWER      Based on total_inventory_value: 0.0
  [1 tool call(s), ~50 tokens]

hand-checked expected value: 4683.95

--------------------------------------------------------------
Same tool, vague docstring:

Q: What is the total value of the paint inventory we have in stock?
  ANSWER      I don't have a tool that can answer that.
  [0 tool call(s), ~26 tokens]


("I don't have a tool that can answer that.", [], 26)

---
# Part 3 — Break it on purpose
**24–32 min · the most valuable section here**

## What this part does
Anyone can demo an agent that works. Knowing the shape of the failures is what
separates "I saw a talk" from "I can ship this." Three failures, most common
first:

| # | Failure | What you see | Where |
|---|---|---|---|
| 1 | Vague, overlapping tool descriptions | confidently wrong tool, nothing raised | **Exercises 2 & 3** — you run these |
| 2 | A tool returning a plausible wrong value | the error compounds across steps | 👀 demo |
| 3 | No step cap | unbounded loop, latency, and bill | 👀 demo |

⏭️ **`CUT AT 30`:** we do Exercises 2 and 3 together, and I demo failure modes 2
and 3.

### 🔧 EXERCISE 2 — diagnose the wrong pick
**~4 min · run it, fix nothing yet**

**The cell below:** two tools whose vague docstrings overlap on "price"/"cost" —
`get_item_price` (the real price) and `get_shipping_cost` (a flat $9.99 fee) —
then one price question.

**Answer these:**
1. Which tool got called?
2. Which docstring words caused that?
3. Why did the *correct* tool lose, even though its **name** is the better match?

Question 3 is the interesting one. In mock mode it is mechanical: ≥2 question
words must appear **in the docstring** for a tool to be eligible at all, and a
good function name cannot rescue a bad docstring. With a real model the mechanism
is fuzzier and the outcome is the same — a top-3 cause of agent bugs in
production.

In [10]:
# ===========================================================================
# EXERCISE 2 — the wrong tool, for a reason you can see.  (about 4 minutes)
#
# Two tools with vague docstrings that overlap on "price" and "cost", then one
# price question. Not a contrived bug: this is what happens when two people on
# a team each write a one-line docstring without reading the other's.
# ===========================================================================

def get_item_price(sku: str) -> dict:
    """Returns a number for the item."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", "price_usd": row["price"]}
    return {"status": "error", "message": "unknown SKU"}

def get_shipping_cost(sku: str) -> dict:
    """Look up the price of the product for a given SKU."""
    # This returns a FLAT SHIPPING FEE. It has nothing to do with retail price.
    return {"status": "success", "shipping_fee_usd": 9.99}


BROKEN_TOOLS = [get_item_price, get_shipping_cost]

print("Ask for a price. Watch which tool actually gets called.\n")
await run_and_trace(make_agent(BROKEN_TOOLS), "What is the price of SKU HD-2001?")

print("\nHD-2001 costs $129.00. The agent just told you $9.99 and was not")
print("uncertain about it for even a moment. Nothing crashed. No error was")
print("raised. This is what a wrong answer looks like in production.")

Ask for a price. Watch which tool actually gets called.

Q: What is the price of SKU HD-2001?
  STEP 1
    TOOL_CALL   get_shipping_cost({'sku': 'HD-2001'})
    OBSERVATION {'status': 'success', 'shipping_fee_usd': 9.99}
  ANSWER      Based on get_shipping_cost: 9.99
  [1 tool call(s), ~32 tokens]

HD-2001 costs $129.00. The agent just told you $9.99 and was not
uncertain about it for even a moment. Nothing crashed. No error was
raised. This is what a wrong answer looks like in production.


### 🔧 EXERCISE 3 — fix it without touching the logic
**~4 min**

**Your job:** rewrite the two docstrings so the price question reaches the price
tool. **Do not change a single line of logic.**

**Done when:** the trace shows `get_item_price` and the answer contains `129.0`.

**Three rules that actually work:**
1. Say what the function **returns**, in the words a user would use.
2. If a sibling tool is nearby, say what this one is **not** for.
3. Do not reuse the other tool's distinguishing nouns.

In [ ]:
# ===========================================================================
# EXERCISE 3 — fix it by rewriting the docstrings.  (about 4 minutes)
#
# Same two tools as Exercise 2. Change ONLY the two docstrings — not one line
# of logic. Done when the trace calls get_item_price and the answer is 129.0.
#
# Rules that actually work:
#   1. Say what the function RETURNS, in the words a user would use.
#   2. Say what it is NOT for, if a sibling tool is nearby.
#   3. Do not reuse the other tool's distinguishing nouns.

def get_item_price(sku: str) -> dict:
    """Returns a number for the item."""
    # YOUR CODE HERE — replace the docstring sentence above.
    # Make a "price" question land HERE. Say that it returns a retail price in
    # dollars for one unit of a product, given its SKU.
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            return {"status": "success", "price_usd": row["price"]}
    return {"status": "error", "message": "unknown SKU"}

def get_shipping_cost(sku: str) -> dict:
    """Look up the price of the product for a given SKU."""
    # YOUR CODE HERE — replace the docstring sentence above.
    # Make this one STOP attracting price questions. It returns a flat shipping
    # and delivery fee. Do not use the word "price" at all.
    return {"status": "success", "shipping_fee_usd": 9.99}


FIXED_TOOLS = [get_item_price, get_shipping_cost]

await run_and_trace(make_agent(FIXED_TOOLS), "What is the price of SKU HD-2001?")

# You are done when the trace calls get_item_price and the answer is 129.0.

Q: What is the price of SKU HD-2001?
  STEP 1
    TOOL_CALL   get_shipping_cost({'sku': 'HD-2001'})
    OBSERVATION {'status': 'success', 'shipping_fee_usd': 9.99}
  ANSWER      Based on get_shipping_cost: 9.99
  [1 tool call(s), ~32 tokens]


('Based on get_shipping_cost: 9.99', ['get_shipping_cost'], 32)

### 👀 Failure mode 2 — the tool that lies quietly
**`WATCH ONLY` if we are short on time**

**The cell below:** a price tool with a cents/dollars bug (off by 100×) feeds a
*perfectly correct* multiply tool. Truth: 40 × $129.00 = $5,160.00. You get
$51.60.

**Why it matters:** nothing raises. A wrong value enters at step 1 and every later
step treats it as fact — from inside the loop, $1.29 looks exactly as valid as
$129.00. Only a verifier **outside** the agent catches it. That is Part 4.

In [11]:
# ===========================================================================
# FAILURE MODE 2 — a tool that lies quietly, and how the error compounds.
#
# A price tool with a cents/dollars bug feeds a perfectly correct multiply
# tool. Nothing here raises: that is the point. A wrong number enters at step 1
# and every later step treats it as fact.
# ===========================================================================

def get_unit_price_BUGGY(sku: str) -> dict:
    """Return the retail price in dollars that a customer pays for one unit of a product, given its SKU."""
    for row in STORE:
        if row["sku"].upper() == sku.upper():
            # The bug: cents/dollars confusion. Plausible magnitude, wrong value.
            # This is a real class of bug, not a strawman.
            return {"status": "success", "price_usd": round(row["price"] / 100, 2)}
    return {"status": "error", "message": "unknown SKU"}

def multiply_price_by_quantity(price_usd: float, quantity: int) -> dict:
    """Multiply a unit price in dollars by an order quantity to compute the order total in dollars."""
    return {"status": "success", "order_total_usd": round(price_usd * quantity, 2)}


print("Truth: HD-2001 is $129.00, so an order of 40 units = $5,160.00\n")
await run_and_trace(
    make_agent([get_unit_price_BUGGY, multiply_price_by_quantity]),
    "What is the order total in dollars for 40 units of SKU HD-2001?",
    max_llm_calls=4,
)
print("\nStep 1 was wrong by 100x. Step 2 did its job perfectly -- it correctly")
print("multiplied a wrong number -- and produced $51.60 instead of $5,160.00.")
print("No exception. No warning. Nothing to grep for in the logs. The agent")
print("cannot detect this, because from inside the loop $1.29 looks exactly as")
print("valid as $129.00. Only a verifier OUTSIDE the agent can. That is Part 4.")

Truth: HD-2001 is $129.00, so an order of 40 units = $5,160.00

Q: What is the order total in dollars for 40 units of SKU HD-2001?
  STEP 1
    TOOL_CALL   get_unit_price_BUGGY({'sku': 'HD-2001'})
    OBSERVATION {'status': 'success', 'price_usd': 1.29}
  STEP 2
    TOOL_CALL   multiply_price_by_quantity({'price_usd': 1.29, 'quantity': 40})
    OBSERVATION {'status': 'success', 'order_total_usd': 51.6}
  ANSWER      Based on get_unit_price_BUGGY: 1.29 Based on multiply_price_by_quantity: 51.6
  [2 tool call(s), ~90 tokens]

Step 1 was wrong by 100x. Step 2 did its job perfectly -- it correctly
multiplied a wrong number -- and produced $51.60 instead of $5,160.00.
No exception. No warning. Nothing to grep for in the logs. The agent
cannot detect this, because from inside the loop $1.29 looks exactly as
valid as $129.00. Only a verifier OUTSIDE the agent can. That is Part 4.


### 👀 Failure mode 3 — no step cap, and watching the meter run
**`WATCH ONLY` if we are short on time**

**The cell below:** a paginating `search_inventory` tool that is honest, useful,
and **never says "done"** — it always offers a next page. Hard-capped at 10 calls
so nobody burns quota, then it prints cumulative tokens and cost per call.

**Read the shape, not the digits** — the token numbers come from a fake model. The
lessons: history grows every step, so cost grows *faster* than step count, and
every step is another round trip your user waits on.

In [12]:
# ===========================================================================
# FAILURE MODE 3 — no natural stop condition. Watch the cost accumulate.
#
# search_inventory() below always offers one more page, so the agent never
# decides it is finished. It runs until the cap cuts it off, then we print the
# running token/cost meter.
# ===========================================================================

# HARD cap: nobody's quota gets burned in this room. try/finally below means
# the counter prints even if the run raises.
HARD_CAP = 10

def search_inventory(query: str, page: int) -> dict:
    """Search the store inventory for products matching a query and return one page of matching results at a time."""
    # The trap: this tool is honest, useful, and never says "done". It always
    # hands back a next page. A paginating API that the agent has no stopping
    # rule for is one of the most common ways real agents run away.
    return {
        "status": "partial",
        "showing": f"page {page}, more results available",
        "next_page": page + 1,
    }


counter = {"calls": 0, "tokens": 0}

async def run_uncapped(question):
    try:
        _text, trajectory, tokens = await run_and_trace(
            make_agent([search_inventory]), question, max_llm_calls=HARD_CAP
        )
        counter["calls"], counter["tokens"] = len(trajectory), tokens

        print("\n  the meter, per call:")
        per_call = tokens / max(1, len(trajectory))
        for i in range(1, len(trajectory) + 1):
            spent = per_call * i
            # ~$0.30 per 1M tokens, Flash-tier order of magnitude. For SCALE
            # only -- these are estimated tokens from a fake model.
            print(f"    call {i:>2}   cumulative ~{spent:>6.0f} tokens   ~${spent * 3e-7:.6f}")
    finally:
        print(f"\n  STOPPED at {counter['calls']} tool call(s), ~{counter['tokens']} tokens.")
        print(f"  It did not stop because it was finished. It stopped because")
        print(f"  RunConfig(max_llm_calls={HARD_CAP}) cut it off.")
        print("  Note the shape: history grows every step, so each call re-sends a")
        print("  bigger prompt. Cost grows FASTER than the number of steps, and")
        print("  every step is another round trip of latency your user waits for.")

await run_uncapped("Find me every product in the store that might be discounted.")

Q: Find me every product in the store that might be discounted.
  STEP 1
    TOOL_CALL   search_inventory({'query': 'discounted', 'page': 1})
    OBSERVATION {'status': 'partial', 'showing': 'page 1, more results available', 'next_page': 2}
  STEP 2
    TOOL_CALL   search_inventory({'query': 'discounted', 'page': 2})
    OBSERVATION {'status': 'partial', 'showing': 'page 2, more results available', 'next_page': 3}
  STEP 3
    TOOL_CALL   search_inventory({'query': 'discounted', 'page': 3})
    OBSERVATION {'status': 'partial', 'showing': 'page 3, more results available', 'next_page': 4}
  STEP 4
    TOOL_CALL   search_inventory({'query': 'discounted', 'page': 4})
    OBSERVATION {'status': 'partial', 'showing': 'page 4, more results available', 'next_page': 5}
  STEP 5
    TOOL_CALL   search_inventory({'query': 'discounted', 'page': 5})
    OBSERVATION {'status': 'partial', 'showing': 'page 5, more results available', 'next_page': 6}
  STEP 6
    TOOL_CALL   search_inventory({'query':

---
# Part 4 — Eval: two different questions
**32–40 min** · ⏭️ **`CUT AT 30`:** I run this one and walk you through the table.

## What this part does
Scores the agent two ways, because these are *not* the same question:

| Metric | Asks | Catches |
|---|---|---|
| **Final-answer accuracy** | did it get the right answer? | obvious wrongness |
| **Trajectory accuracy** | did it get there the right way? | right-by-accident |

**Watch for the case that PASSES on answer and FAILS on trajectory.** That case is
the whole point: an agent that is right by accident passes an answer-only eval,
ships, and breaks silently three weeks later when the tool it accidentally leaned
on changes its return shape.

## What the cell below does
Loads 10 cases from `eval_set.json` (each with an expected answer **and** an
expected tool sequence) → runs the agent on all of them with the six `EVAL_TOOLS`
→ prints a scored table, then the "right answer, wrong path" list.

This works because the dataset is 20 rows and ground truth is **cheap**. Hold that
thought — it is where Part 5 starts.

In [13]:
# ===========================================================================
# PART 4 — Eval. Score the answer AND the path taken, separately.
#
#   score_answer()      did the final text contain the expected strings?
#   score_trajectory()  did the tool sequence match, in order?
# ...then run all 10 cases from eval_set.json and print the table.
#
# EVAL_TOOLS comes from Cell 0 (Part 2's four tools, plus find_product_by_name
# and total_inventory_value), so this cell runs on its own.
# ===========================================================================

with open("eval_set.json") as fh:
    EVAL = json.load(fh)


def score_answer(final_text, expected_substrings):
    """The cheap verifier. Every expected string must appear in the answer.

    Crude on purpose. The reason agents work at all in a domain is that you can
    write a check like this. If you cannot, you do not have an eval -- you have
    a vibe.
    """
    return all(s.lower() in (final_text or "").lower() for s in expected_substrings)

def score_trajectory(actual, expected):
    """Exact tool sequence match. Order matters."""
    return list(actual) == list(expected)


results = []
for case in EVAL["cases"]:
    try:
        final_text, trajectory, tokens = await run_and_trace(
            make_agent(EVAL_TOOLS), case["question"], max_llm_calls=6, show=False
        )
    except Exception as e:                      # one bad case must not kill the table
        final_text, trajectory, tokens = f"ERROR {type(e).__name__}: {e}", [], 0
    results.append({
        "id": case["id"],
        "answer_ok": score_answer(final_text, case["expected_answer_contains"]),
        "traj_ok": score_trajectory(trajectory, case["expected_trajectory"]),
        "expected_traj": " -> ".join(case["expected_trajectory"]) or "(none)",
        "actual_traj": " -> ".join(trajectory) or "(none)",
        "tokens": tokens,
    })

# ---- the table -------------------------------------------------------------
print(f"{'case':<38} {'answer':<8} {'traj':<7} actual trajectory")
print("-" * 100)
for r in results:
    print(f"{r['id']:<38} {'PASS' if r['answer_ok'] else 'FAIL':<8} "
          f"{'PASS' if r['traj_ok'] else 'FAIL':<7} {r['actual_traj']}")

n = len(results)
answer_score = sum(r["answer_ok"] for r in results)
traj_score = sum(r["traj_ok"] for r in results)
both = sum(r["answer_ok"] and r["traj_ok"] for r in results)
lucky = [r for r in results if r["answer_ok"] and not r["traj_ok"]]

print("-" * 100)
print(f"final-answer accuracy : {answer_score}/{n}  ({100*answer_score//n}%)")
print(f"trajectory accuracy   : {traj_score}/{n}  ({100*traj_score//n}%)")
print(f"both correct          : {both}/{n}")
print(f"total tokens          : ~{sum(r['tokens'] for r in results)}")

if lucky:
    print()
    print("RIGHT ANSWER, WRONG PATH -- the dangerous column:")
    for r in lucky:
        print(f"  {r['id']}")
        print(f"     expected: {r['expected_traj']}")
        print(f"     actual:   {r['actual_traj']}")
    print()
    print("  These pass an answer-only eval and would ship. They are the cases")
    print("  that break silently later, when the tool they accidentally leaned")
    print("  on changes its return shape. Answer accuracy alone hides them.")

case                                   answer   traj    actual trajectory
----------------------------------------------------------------------------------------------------
01-price-by-sku                        PASS     PASS    get_product_details
02-aisle-by-sku                        PASS     PASS    get_product_details
03-category-listing                    PASS     PASS    list_products_in_category
04-order-total                         PASS     PASS    calculate_order_total
05-stock-right-answer-wrong-trajectory PASS     FAIL    get_product_details
06-inventory-value                     PASS     PASS    total_inventory_value
07-search-by-name                      PASS     PASS    find_product_by_name
08-electrical-listing                  PASS     PASS    list_products_in_category
09-stock-of-fasteners-item             PASS     PASS    check_stock_quantity
10-unanswerable                        PASS     PASS    (none)
------------------------------------------------------------

### Reading that table

- **Answer accuracy alone is not a passing grade.** Compare the two columns.
- `05-stock-right-answer-wrong-trajectory` reaches the right number through
  `get_product_details` instead of `check_stock_quantity`. Both return stock
  quantity today. Only one of them is *supposed* to.
- `10-unanswerable` checks that the agent **declines**. Inventing a supplier name
  is worse than erroring — a plausible fabrication survives review.
- Scores are byte-for-byte reproducible in mock mode, so you can tell a regression
  from noise.

A real eval set is 100–1000 cases and lives in CI. The mechanism is exactly what
you just ran.

---
# Part 5 — Where agents actually work
**40–45 min · no code · sit back**

## The one question that predicts success
**Can you cheaply verify the output?** Not "is the model good." Can you write a
check — like `score_answer` in Part 4 — that catches a wrong answer for less than
the cost of producing it?

- **Yes** → agents work: wrong answers get caught and retried.
- **No** → you are shipping unverified generated output and calling it automation.

## Where they work, and where they don't
| ✅ Works when | ❌ Fails when |
|---|---|
| **A cheap verifier exists** — tests pass, the code compiles, the number reconciles | **No ground-truth signal.** You can't eval it, improve it, or detect a regression. That is a demo, not a system. |
| **Retry is cheap and safe** — pennies, and it breaks nothing | **Silent error accumulation** (failure mode 2). Every step is plausible, nothing alarms, and step 5 is confidently wrong. |
| **Step count is small and bounded** — 2–5 calls, not 50 | **Latency budgets under a second.** Every step is a round trip. |
| **A human sits at a natural checkpoint** — reviewing a diff, approving a draft | **Irreversible actions.** Sending the email. Issuing the refund. Dropping the table. |
| **Being wrong is cheap,** or obvious immediately | **Cost that multiplies invisibly.** History grows each step, so 10× the steps is well over 10× the bill. |

## What to do on Monday
1. Write the **verifier first**. If you cannot, stop — that is the finding.
2. Start with a **fixed workflow**. Add agency only where you provably cannot
   enumerate the steps.
3. **Cap the loop** on day one. `max_llm_calls` is a correctness requirement, not
   a production-hardening task.
4. Treat **docstrings as prompts**, and review them like prompts.
5. Score **trajectory as well as answer**, or you will ship agents that are right
   by accident.

**The honest summary:** an agent is a while loop with a model in the condition. A
good tool when you can check its work; a liability when you cannot.

---
## Take it home — running this against a real model

Everything you ran used the mock. Three steps to swap in Gemini, and to feel the
difference — a real model generalizes from what your docstring *means*, not from
which words happen to overlap:

1. **Free AI Studio API key**, no credit card:
   [aistudio.google.com/apikey](https://aistudio.google.com/apikey) → sign in →
   **Create API key** → **Create API key in new project** → copy it.
2. **`cp .env.example .env`** and paste it in as `GOOGLE_API_KEY=your-key-here`.
   `.env` is gitignored; do not commit it.
3. **In Cell 0, set `MODE = "live"`** and re-run from the top.

Everything else is byte-identical. The only line that changes between a fake model
and Gemini is `MODE`.

**Two things worth trying once you are live:**
- Re-run Exercise 1 with a docstring built entirely from *synonyms* of the
  question's words. The mock misses it; a real model usually does not. That gap is
  word matching vs. language understanding.
- Re-run Part 4 a few times. Scores may **move** between runs even at temperature
  0. Non-determinism is a property of the real thing, and it is why a fixed eval
  set matters more, not less.

Free-tier keys are rate-limited. Part 4 makes ~10 calls; on a quota error, wait a
minute or set `MODEL_AI_STUDIO = "gemini-3.5-flash-lite"`.

Repo, slides, and these notebooks: see the README. Questions welcome — find me
after the talk.